In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [10]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "kiwi-com-flight-search": {
            "url": "https://mcp.kiwi.com",
            "transport": "http",
        }
    }
)

tools = await client.get_tools()

In [14]:
tools

[StructuredTool(name='search-flight', description='# Search for flights\n\nSearches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.\n\n## Result shape\n\nReturns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:\n- `price` (number) and `priceFormatted` (e.g. "123 EUR")\n- `totalDurationSeconds`\n- `bookingUrl` — the link to book the flight\n- `imageId` — destination city id for a hero photo (https://images.kiwi.com/photos/600x600/{imageId}.jpg)\n- `baggage` — total included baggage across all travelers: `{ personalItem, cabinBag, checkedBag }` (counts)\n- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (loc

In [17]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("kiwi-com-flight-search")

# # get prompts
# prompt = await client.get_prompt("kiwi-com-flight-search", "prompt")
# prompt = prompt[0].content

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    #system_prompt=prompt
)

In [12]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Flight options from Austin, TX to San Francisco")]},
    config=config
)

In [13]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Flight options from Austin, TX to San Francisco', additional_kwargs={}, response_metadata={}, id='291a64a3-d4cc-438d-9c2e-c17dd45aab16'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search-flight', 'arguments': '{"departureDate": "20/06/2025", "flyTo": "San Francisco", "flyFrom": "Austin, TX"}'}, '__gemini_function_call_thought_signatures__': {'call_102660': 'EnEKbwFpFH0TUug+2C0uLxk+q1dgiNWNE7MASqkCp6NNT6DgeyTbXyxvCDhWE2b/L/IhVhQLEc2YI3JNVMKo+5GR0oseYl150hL0BVrHsavYheKB4hxhf4KPLjCwhTk7Ni1v7xLpyP2Do0Y6V4MYV2Fjxw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b5bf-72d1-77d0-9f77-2a690cc5e6cd-0', tool_calls=[{'name': 'search-flight', 'args': {'departureDate': '20/06/2025', 'flyTo': 'San Francisco', 'flyFrom': 'Austin, TX'}, 'id': 'call_102660', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [ ]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
)

In [ ]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)